# 02 - Ingestão Bronze

Baixa os arquivos de dados abertos da ANEEL para o volume de landing e registra astabelas da camada Bronze, preservando o dado como publicado na origem.

O que este notebook faz:
1. Resolve, pela API CKAN do portal, a URL e a licença de cada recurso a partir do identificador do conjunto de dados. As URLs não ficam fixas no código.
2. Cria as subpastas de landing que ainda não existem.
3. Baixa cada arquivo para o volume, pulando o que já está presente com o tamanho correto.
4. Registra as tabelas Bronze em formato Delta, acrescentando colunas de linhagem.
5. Grava uma tabela de controle com licença, URL, tamanho e contagem de linhas por recurso.
6. Valida as contagens.

Princípio da camada Bronze: nenhuma transformação de conteúdo. Tipos, nomes de coluna e valores chegam como estão na origem. Harmonização é trabalho da Silver — incluindo a diferença de leiaute da base de ocorrências emergenciais, que por isso gera duas tabelas distintas, `emergency_occurrences_v1` e `emergency_occurrences_v2`.

In [0]:
dbutils.library.restartPython()

## 1. Configuração

In [0]:
import json
import os
import sys
import time
from datetime import datetime, timezone

import requests
from pyspark.sql import functions as F


# The repository root is the parent of the notebooks folder, so that `src` is importable
REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import (
    ANEEL_API_URL,
    CATALOG,
    DATASETS,
    INGESTION_LOG_TABLE,
    LANDING_PATH,
    SCHEMA_BRONZE,
    SOURCE_FOLDERS,
)

BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}"

# Streaming parameters for the download step
CHUNK_BYTES = 8 * 1024 * 1024
REQUEST_TIMEOUT = (30, 600)
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 10

# Single ingestion timestamp for the whole run, so every row of this batch shares it
RUN_TIMESTAMP = datetime.now(timezone.utc)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_BRONZE}")

print(f"Catalogo.......: {BRONZE}")
print(f"Landing........: {LANDING_PATH}")
print(f"Conjuntos......: {len(DATASETS)}")
print(f"Tabelas Bronze.: {sum(len(d['tables']) for d in DATASETS.values())}")
print(f"Execucao (UTC).: {RUN_TIMESTAMP:%Y-%m-%d %H:%M:%S}")

In [0]:
def human_size(num_bytes):
    """Format a byte count as a human readable string."""
    if num_bytes is None:
        return "n/d"
    value = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(value) < 1024.0:
            return f"{value:,.1f} {unit}"
        value /= 1024.0
    return f"{value:,.1f} PB"

## 2. Metadados e licença via API CKAN
O portal da ANEEL expõe uma API CKAN. A chamada `package_show` devolve, para cada conjunto de dados, o título, a licença e a lista de recursos com suas URLs de download. Resolver a URL em tempo de execução, em vez de fixá-la no código, tem duas vantagens: a ingestão continua funcionando se a ANEEL republicar um recurso sob novo identificador, e a licença de cada conjunto passa a ser capturada automaticamente, que é a evidência exigida pelo critério de Coleta.

In [0]:
def fetch_package(dataset_id):
    """Read one CKAN package and return its title, licence and resource index."""
    response = requests.get(
        f"{ANEEL_API_URL}/package_show",
        params={"id": dataset_id},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    payload = response.json()
    if not payload.get("success"):
        raise RuntimeError(f"CKAN retornou success=False para {dataset_id}")

    package = payload["result"]

    # Index resources by file name taken from the download URL, which is how the
    # source catalog in config.py refers to them.
    resources = {}
    for resource in package.get("resources", []):
        url = resource.get("url") or ""
        file_name = url.rsplit("/", 1)[-1]
        if file_name:
            resources[file_name] = {
                "url": url,
                "format": resource.get("format"),
                "size": resource.get("size"),
                "last_modified": resource.get("last_modified") or resource.get("created"),
            }

    return {
        "dataset_id": dataset_id,
        "title": package.get("title"),
        "license_title": package.get("license_title"),
        "license_id": package.get("license_id"),
        "license_url": package.get("license_url"),
        "metadata_modified": package.get("metadata_modified"),
        "resources": resources,
    }

In [0]:
# Resolve every dataset once and keep the result in memory for the whole notebook
packages = {}
missing_resources = []

for key, spec in DATASETS.items():
    package = fetch_package(spec["dataset_id"])
    packages[key] = package

    expected = [f for files in spec["tables"].values() for f in files]
    absent = [f for f in expected if f not in package["resources"]]
    if absent:
        missing_resources.append((key, absent))

    print(f"{key:<28} {package['license_id'] or 'n/d':<12} "
          f"{len(package['resources']):>3} recursos   {package['title']}")

if missing_resources:
    print()
    print("ATENCAO - recursos nao encontrados no portal:")
    for key, absent in missing_resources:
        for file_name in absent:
            print(f"  {key}: {file_name}")

In [0]:
# Licence summary, one row per dataset. This is the evidence for the Coleta criterion.
license_rows = [
    {
        "dataset_key": key,
        "dataset_id": package["dataset_id"],
        "title": package["title"],
        "license_id": package["license_id"],
        "license_title": package["license_title"],
        "metadata_modified": package["metadata_modified"],
    }
    for key, package in packages.items()
]

display(spark.createDataFrame(license_rows))

## 3. Pastas de landing
O notebook de setup criou as subpastas conhecidas naquele momento. Duas fontes foram acrescentadas depois — ocorrências emergenciais e qualidade do atendimento comercial — e a criação aqui é idempotente, de modo que executar de novo não quebra nada.

In [0]:
for folder, description in SOURCE_FOLDERS.items():
    path = f"{LANDING_PATH}/{folder}"
    existed = os.path.isdir(path)
    os.makedirs(path, exist_ok=True)
    status = "ja existia" if existed else "criada"
    print(f"{folder:<30} {status:<12} {description}")

## 4. Download para o volume

O download é feito em fluxo, gravando direto no volume, sem carregar o arquivo na memória do driver. Um arquivo já presente com o mesmo tamanho do recurso é pulado. Essa verificação existe por causa da cota diária do Databricks Free Edition: se o compute for desligado no meio da carga, basta executar a célula novamente no dia seguinte e apenas o que faltava é transferido.

In [0]:
def download_resource(url, destination):
    """Stream one resource into the landing volume and return the byte count."""
    temp_path = f"{destination}.part"

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT) as response:
                response.raise_for_status()
                expected = response.headers.get("Content-Length")
                expected = int(expected) if expected is not None else None

                written = 0
                started = time.time()
                with open(temp_path, "wb") as handle:
                    for chunk in response.iter_content(chunk_size=CHUNK_BYTES):
                        if chunk:
                            handle.write(chunk)
                            written += len(chunk)

            if expected is not None and written != expected:
                raise IOError(f"tamanho divergente: recebido {written}, esperado {expected}")

            os.replace(temp_path, destination)
            elapsed = max(time.time() - started, 0.001)
            print(f"    {human_size(written)} em {elapsed:,.0f}s "
                  f"({written / elapsed / (1024 * 1024):,.1f} MB/s)")
            return written

        except (requests.RequestException, IOError) as error:
            print(f"    tentativa {attempt}/{MAX_RETRIES} falhou: {error}")
            if os.path.exists(temp_path):
                os.remove(temp_path)
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)

    raise RuntimeError(f"download falhou apos {MAX_RETRIES} tentativas: {url}")

In [0]:
# One entry per downloaded resource, reused by the Bronze registration step below
downloads = []

total_files = sum(len(f) for s in DATASETS.values() for f in s["tables"].values())
position = 0
batch_started = time.time()


def stamp():
    """Cluster wall-clock timestamp used on progress lines."""
    return datetime.now().strftime("%H:%M:%S")


for key, spec in DATASETS.items():
    package = packages[key]
    folder = spec["folder"]
    print(f"\n{key}", flush=True)

    for table_name, file_names in spec["tables"].items():
        for file_name in file_names:
            position += 1
            prefix = f"  [{position:>2}/{total_files}]"

            resource = package["resources"].get(file_name)
            if resource is None:
                print(f"{prefix} {file_name}: AUSENTE no portal, pulando", flush=True)
                continue

            destination = f"{LANDING_PATH}/{folder}/{file_name}"
            remote_size = int(resource["size"]) if resource.get("size") else None

            if os.path.exists(destination):
                local_size = os.path.getsize(destination)
                if remote_size is None or local_size == remote_size:
                    print(f"{prefix} {file_name}: ja presente "
                          f"({human_size(local_size)}) - {stamp()}", flush=True)
                    downloads.append({
                        "dataset_key": key, "table": table_name, "file": file_name,
                        "url": resource["url"], "bytes": local_size,
                        "last_modified": resource.get("last_modified"),
                    })
                    continue
                print(f"{prefix} {file_name}: tamanho divergente, rebaixando", flush=True)

            print(f"{prefix} {file_name}: inicio {stamp()}", flush=True)
            written = download_resource(resource["url"], destination)
            print(f"{prefix} {file_name}: CONCLUIDO {stamp()} "
                  f"({human_size(written)}; lote a {time.time() - batch_started:,.0f}s)",
                  flush=True)

            downloads.append({
                "dataset_key": key, "table": table_name, "file": file_name,
                "url": resource["url"], "bytes": written,
                "last_modified": resource.get("last_modified"),
            })

print(f"\nRecursos no volume: {len(downloads)} de {total_files} "
      f"({human_size(sum(d['bytes'] for d in downloads))}) "
      f"em {time.time() - batch_started:,.0f}s", flush=True)

### 4.1. Normalização de precisão temporal

A carga da camada Bronze falhou na primeira execução com `PARQUET_TYPE_ILLEGAL: Illegal
Parquet type: INT64 (TIMESTAMP(NANOS,false))`. Ao menos um dos arquivos publicados pela
ANEEL grava colunas de data e hora com precisão de nanossegundos, e o Spark suporta
apenas até microssegundos.

A precisão não é intencional. É assinatura do programa que gerou o arquivo: o tipo de
data padrão do pandas é `datetime64[ns]`, e gravar esse objeto em Parquet sem coerção
explícita produz `TIMESTAMP(NANOS)`. Os valores em questão são datas de competência e
horários de ocorrência, cuja precisão real não desce do segundo. Não há informação
abaixo do microssegundo a preservar.

Existe uma configuração que faria o Spark ler essas colunas como inteiro,
`spark.sql.legacy.parquet.nanosAsLong`. Ela não está disponível no compute serverless
da Free Edition, que bloqueia configurações legadas. A restrição de ambiente escolhida
para o projeto, portanto, teve consequência direta sobre o pipeline.

A solução adotada converte o arquivo antes da leitura, gravando uma cópia com
microssegundos em uma subpasta `_normalized` dentro da mesma pasta de landing. Três
cuidados tornam isso compatível com o papel da camada Bronze:

O arquivo original permanece no volume, intocado, ao lado da cópia. A evidência da
coleta continua disponível para conferência, que é a razão de a Bronze existir.

A conversão é declarada e reproduzível. Ela altera apenas a unidade de precisão do
tipo, não o valor, e o código que a executa está versionado junto do pipeline.

A leitura é transparente: `read_source_file` usa a cópia normalizada quando ela existe
e o original quando não existe, de modo que o restante do notebook não precisa saber
quais arquivos foram afetados.

A conversão percorre o arquivo row group por row group, em vez de carregá-lo inteiro.
O maior arquivo do conjunto tem quase 24 milhões de linhas, e lê-lo de uma vez na
memória do driver não caberia na Free Edition.

Esse é um achado de qualidade sobre a fonte, não sobre o pipeline: um dado público
distribuído em formato que a ferramenta mais comum do ecossistema não consegue abrir
sem intervenção. A seção de qualidade de dados retoma o ponto.

In [0]:
import pyarrow as pa
import pyarrow.parquet as pq

# Landing sub-folder holding Spark-readable copies of files whose original schema
# Spark cannot read. The original file is always kept next to it.
NORMALIZED_DIR = "_normalized"


def nanosecond_columns(path):
    """Return the names of TIMESTAMP(NANOS) columns of a Parquet file."""
    schema = pq.ParquetFile(path).schema_arrow
    return [
        name for name, dtype in zip(schema.names, schema.types)
        if pa.types.is_timestamp(dtype) and dtype.unit == "ns"
    ]


def normalize_timestamps(source_path, target_path):
    """Rewrite a Parquet file casting nanosecond timestamps to microseconds.

    Spark rejects TIMESTAMP(NANOS). The nanosecond unit here is an artifact of the
    writer (pandas datetime64[ns]); the values carry no sub-microsecond information,
    so the cast is lossless in practice. Row groups are streamed one at a time to
    keep driver memory bounded.
    """
    parquet_file = pq.ParquetFile(source_path)
    source_schema = parquet_file.schema_arrow

    fields = [
        field.with_type(pa.timestamp("us", tz=field.type.tz))
        if pa.types.is_timestamp(field.type) and field.type.unit == "ns"
        else field
        for field in source_schema
    ]
    target_schema = pa.schema(fields)

    writer = pq.ParquetWriter(target_path, target_schema, compression="snappy")
    try:
        for batch in parquet_file.iter_batches(batch_size=200_000):
            writer.write_table(pa.Table.from_batches([batch]).cast(target_schema))
    finally:
        writer.close()

In [0]:
# Files whose landing copy had to be normalised, keyed by file name
normalized_files = {}

for key, spec in DATASETS.items():
    if spec["format"] != "parquet":
        continue

    folder = spec["folder"]
    for file_names in spec["tables"].values():
        for file_name in file_names:
            source = f"{LANDING_PATH}/{folder}/{file_name}"
            if not os.path.exists(source):
                continue

            columns = nanosecond_columns(source)
            if not columns:
                continue

            target_dir = f"{LANDING_PATH}/{folder}/{NORMALIZED_DIR}"
            os.makedirs(target_dir, exist_ok=True)
            target = f"{target_dir}/{file_name}"

            if not os.path.exists(target):
                print(f"{file_name}: convertendo {columns}", flush=True)
                normalize_timestamps(source, target)
            else:
                print(f"{file_name}: versao normalizada ja existe", flush=True)

            normalized_files[file_name] = columns

if normalized_files:
    print(f"\nArquivos normalizados: {len(normalized_files)}")
    for file_name, columns in normalized_files.items():
        print(f"  {file_name}: {', '.join(columns)}")
else:
    print("Nenhum arquivo com timestamp em nanossegundos.")

## 5. Registro das tabelas Bronze:

Cada tabela recebe duas colunas de linhagem: `_source_file`, com o nome do arquivo de origem de cada linha, e `_ingested_at`, com o instante desta execução. Quando uma tabela é composta por vários arquivos, os esquemas são comparados antes da união. Divergência interrompe a carga em vez de unir silenciosamente — é justamente o que teria acontecido com as ocorrências emergenciais se os quatro anos estivessem na mesma tabela. O CSV do PDD é lido com todas as colunas como texto. Valores com vírgula decimal einteiros com zeros à esquerda seriam corrompidos por inferência de tipo, e a conversão é decisão da Silver, com a regra registrada.

In [0]:
def read_source_file(spec, folder, file_name):
    """Read one landing file into a DataFrame, preserving the published content."""
    # Prefer the normalised copy when the original schema is unreadable by Spark
    normalized = f"{LANDING_PATH}/{folder}/{NORMALIZED_DIR}/{file_name}"
    path = normalized if os.path.exists(normalized) else f"{LANDING_PATH}/{folder}/{file_name}"
    ...

    if spec["format"] == "parquet":
        frame = spark.read.parquet(path)
    elif spec["format"] == "csv":
        options = spec.get("csv_options", {})
        reader = spark.read.format("csv").option("inferSchema", "false")
        for option, value in options.items():
            reader = reader.option(option, value)
        frame = reader.load(path)
    else:
        raise ValueError(f"formato nao suportado: {spec['format']}")

    return frame.withColumn("_source_file", F.lit(file_name)) \
                .withColumn("_ingested_at", F.lit(RUN_TIMESTAMP).cast("timestamp"))


def assert_same_schema(frames, file_names, table_name):
    """Abort when files that compose one table do not share the same schema."""
    reference = frames[0].schema
    for frame, file_name in zip(frames[1:], file_names[1:]):
        if frame.schema != reference:
            only_reference = set(reference.names) - set(frame.schema.names)
            only_current = set(frame.schema.names) - set(reference.names)
            raise ValueError(
                f"esquemas divergentes em {table_name}: "
                f"{file_names[0]} x {file_name}. "
                f"Apenas no primeiro: {sorted(only_reference)}. "
                f"Apenas no segundo: {sorted(only_current)}."
            )

In [0]:
# Write one Delta table per entry of the source catalog
table_stats = {}

for key, spec in DATASETS.items():
    folder = spec["folder"]

    for table_name, file_names in spec["tables"].items():
        present = [f for f in file_names
                   if os.path.exists(f"{LANDING_PATH}/{folder}/{f}")]
        if not present:
            print(f"{table_name}: nenhum arquivo no volume, pulando")
            continue

        frames = [read_source_file(spec, folder, f) for f in present]
        assert_same_schema(frames, present, table_name)

        combined = frames[0]
        for frame in frames[1:]:
            combined = combined.unionByName(frame)

        full_name = f"{BRONZE}.{table_name}"
        combined.write.mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(full_name)

        rows = spark.table(full_name).count()
        columns = len(spark.table(full_name).columns)
        table_stats[table_name] = {"rows": rows, "columns": columns, "files": len(present)}

        print(f"{table_name:<32} {rows:>14,} linhas  {columns:>3} colunas  "
              f"{len(present)} arquivo(s)")

## 6. Tabela de controle da ingestão:

Uma linha por recurso carregado, com licença, URL de origem, data de atualização no portal, tamanho e tabela de destino. É a linhagem da camada Bronze e a evidência documental da etapa de coleta.

In [0]:
log_rows = []
for entry in downloads:
    package = packages[entry["dataset_key"]]
    log_rows.append({
        "dataset_key": entry["dataset_key"],
        "dataset_id": package["dataset_id"],
        "dataset_title": package["title"],
        "license_id": package["license_id"],
        "license_title": package["license_title"],
        "resource_file": entry["file"],
        "resource_url": entry["url"],
        "resource_last_modified": entry["last_modified"],
        "resource_bytes": entry["bytes"],
        "target_table": f"{BRONZE}.{entry['table']}",
        "ingested_at": RUN_TIMESTAMP,
    })

log_frame = spark.createDataFrame(log_rows)
log_frame.write.mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable(f"{BRONZE}.{INGESTION_LOG_TABLE}")

display(spark.table(f"{BRONZE}.{INGESTION_LOG_TABLE}"))

## 7. Validação:

Confere o que foi efetivamente persistido no catálogo e compara com o esperado. Os números de linha devem bater com o perfil levantado nos arquivos locais antes da carga.

In [0]:
tables = spark.sql(f"SHOW TABLES IN {BRONZE}").collect()
print(f"Tabelas em {BRONZE}: {len(tables)}\n")

total_rows = 0
for table_name, stats in sorted(table_stats.items()):
    total_rows += stats["rows"]
    print(f"{table_name:<32} {stats['rows']:>14,} linhas  {stats['columns']:>3} colunas")

print(f"\n{'TOTAL':<32} {total_rows:>14,} linhas")

expected_tables = {t for spec in DATASETS.values() for t in spec["tables"]}
persisted = {row["tableName"] for row in tables}
absent = expected_tables - persisted
if absent:
    print(f"\nATENCAO - tabelas esperadas e nao persistidas: {sorted(absent)}")
else:
    print("\nTodas as tabelas esperadas foram persistidas.")

In [0]:
# Row count per source file, to confirm that every year landed in its table
display(
    spark.sql(f"""
        SELECT target_table,
               resource_file,
               resource_bytes,
               resource_last_modified
        FROM {BRONZE}.{INGESTION_LOG_TABLE}
        ORDER BY target_table, resource_file
    """)
)

## 8. Autoavaliação desta etapa

**O que esta etapa entregou.** Catorze recursos resolvidos pela API CKAN a partir do
identificador do conjunto, com licença e data de atualização capturadas no mesmo passo,
cerca de 1,7 GB transferidos para o volume de landing e nove tabelas Delta registradas
com colunas de linhagem. A tabela de controle `_ingestion_log` liga cada recurso à sua
URL de origem, ao tamanho e à tabela de destino.

A resolução por identificador, em vez de URL fixa, fechou de graça uma pendência que
estava aberta desde a definição do escopo: a licença de cada conjunto passou a ser lida
do portal a cada execução, em vez de ser conferida manualmente.

**O que mudou no caminho.** Três coisas.

O catálogo de fontes cresceu depois do notebook de setup. As pastas de ocorrências
emergenciais e de qualidade do atendimento comercial não existiam no volume, porque
foram acrescentadas ao escopo depois que o setup rodou. A criação das subpastas passou
a ser feita aqui, de forma idempotente, em vez de depender de o setup estar atualizado.

A base de ocorrências emergenciais virou duas tabelas. O arquivo de 2026 é publicado sob
um padrão de nomenclatura diferente dos anos anteriores, com vinte e três colunas contra
quinze e toda a tipagem nativa convertida em texto. Unir os quatro anos numa tabela só
exigiria decidir o de-para antes de examinar os dados, e a Bronze não é o lugar dessa
decisão. Daí `emergency_occurrences_v1` e `emergency_occurrences_v2`, com a harmonização
adiada para a Silver. A guarda de esquema que aborta a união quando os arquivos divergem
existe por causa desse caso.

A carga completa coube na cota. O plano original previa filtrar os arquivos localmente
e subir apenas o recorte, caso o download direto não coubesse no limite diário da Free
Edition. Não foi necessário: o pipeline permanece automatizado de ponta a ponta, sem a
etapa manual que teria de ser justificada no trabalho.

**O obstáculo não previsto.** O registro Bronze falhou com `PARQUET_TYPE_ILLEGAL` num
arquivo com precisão de nanossegundos, que o Spark não lê. A configuração que permitiria
a leitura é bloqueada no compute serverless. Foi o primeiro ponto em que a restrição de
ambiente escolhida para o projeto teve consequência concreta sobre o código, e a solução
— converter o arquivo preservando o original — está descrita na seção 4.1.

**O que fica em aberto.** O de-para entre os dois leiautes de ocorrências, que decide se
a série de trinta e seis meses pode ser unida. A escolha entre as duas bases de serviços
comerciais, que medem o mesmo tema em granularidades diferentes. E a conferência das
contagens desta camada contra o perfil levantado nos arquivos antes da carga.

**O que eu faria diferente.** Perfilar o esquema de todos os arquivos localmente, e não
apenas o das ocorrências emergenciais. O problema dos nanossegundos é visível no rodapé
do Parquet e teria aparecido em segundos numa inspeção local, antes de consumir cota de
compute e interromper a carga na sétima tabela. O perfil que fiz respondeu quantas linhas
e quantas colunas cada arquivo tem; faltou perguntar de que tipo são.